Cosyvoice TTS

In [ ]:
import sys
sys.path.append('/root/LLMs/CosyVoice')
sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')

from cosyvoice.cli.cosyvoice import CosyVoice, CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio


cosyvoice = CosyVoice2('/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False)

prompt_speech_16k = load_wav('/root/LLMs/data/ZH_B00000_S00590_W000016.mp3', 16000)
promp_text = '当我们不找借口的时候我们就是在承担责任，一个人只有承担了才能真正成长。'

# for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', promp_text, prompt_speech_16k, stream=False)):
#     torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

2025-01-07 21:52:04,254 - modelscope - INFO - PyTorch version 2.3.1+cu121 Found.
2025-01-07 21:52:04,255 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2025-01-07 21:52:04,276 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 aa680ed5ee387dd4d8d6eda98a84dc8a and a total number of 980 components indexed
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

# 读取JSON文件
instruction_file = "/root/LLMs/data/instruction_300_text.json"

try:
    with open(instruction_file, "r", encoding="utf-8") as file:
        data = json.load(file)
except FileNotFoundError:
    print(f"文件 {instruction_file} 未找到。")
    exit(1)
except json.JSONDecodeError:
    print(f"文件 {instruction_file} 格式不正确。")
    exit(1)

# 提取所有input内容
output = [item["output"] for item in data]

# # 打印所有input内容
# for i, input_text in enumerate(output, start=1):
#     print(f"{i}. {input_text}")

In [3]:
tts_texts = output

for idx, tts_text in enumerate(tts_texts):
    results = cosyvoice.inference_zero_shot(tts_text, promp_text, prompt_speech_16k, stream=False)

    # 保存合成的语音到文件
    for i, result in enumerate(results):
        if "tts_speech" in result:
            torchaudio.save(
                f"/root/LLMs/data/glm_flow/Q_{idx}_{i}.wav", result["tts_speech"], 22050
            )
        else:
            print(f"警告：结果中没有找到'tts_speech'键，跳过第{idx}_{i}个结果")


  0%|          | 0/1 [00:00<?, ?it/s]

2025-01-07 17:26:03,607 INFO synthesis text 我是来自纳友科技的智能语音助手，很高兴为您服务。
We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)
2025-01-07 17:26:08,907 INFO yield speech len 6.16, rtf 0.8603675411893176
  0%|          | 0/1 [00:00<?, ?it/s]2025-01-07 17:26:09,030 INFO synthesis text 我是纳友科技最新推出的智能语音助手模型。
2025-01-07 17:26:12,157 INFO yield speech len 5.6, rtf 0.5584407704217094
  0%|          | 0/1 [00:00<?, ?it/s]2025-01-07 17:26:12,282 INFO synthesis text 我是纳友科技设计的助手，可以帮助您处理各种任务。
2025-01-07 17:26:16,629 INFO yield speech len 5.76, rtf 0.754636029402415
  0%|          | 0/1 [00:00<?, ?it/s]2025-01-07 17:26:16,749 INFO synthesis text 我来自纳友科技，可以帮您查询信息、管理日程等。
2025-01-07 17:26:22,106 INFO yield speech len 7.0, rtf 0.7651830400739398
  0%|          | 0/1 [00:00<?, ?it/s]2025-01-07 17:26:22,226 INFO

In [2]:
import os
import json

def generate_audio_paths(json_path, audio_dir):
    """
    根据 JSON 文件中的顺序生成对应的音频文件路径列表
    :param json_path: JSON 文件路径
    :param audio_dir: 音频文件目录
    :return: 音频文件路径列表
    """
    # 读取 JSON 文件
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 生成音频文件路径列表
    audio_paths = []
    for idx in range(len(data)):
        audio_filename = f"Q_{idx}_0.wav"
        audio_path = os.path.join(audio_dir, audio_filename)
        audio_paths.append(audio_path)
    
    return audio_paths



In [3]:
# 定义 JSON 文件路径和音频文件目录
json_path = '/root/LLMs/data/instruction_300_text.json'
audio_dir = '/root/LLMs/data/glm_flow'

audio_paths = generate_audio_paths(json_path, audio_dir)

audio_paths = audio_paths[:10]

Cosyvoice 提取 Speaker Embedding

In [3]:
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import onnxruntime
# import numpy as np

def extract_embedding(audio_path, onnx_model_path='/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/campplus.onnx'):
    """
    提取单个音频文件的 embedding
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: embedding (numpy array)
    """
    # 加载音频文件
    audio, sample_rate = torchaudio.load(audio_path)

    # 如果采样率不是16000，则进行重采样
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)

    # 计算音频的FBank特征
    feat = kaldi.fbank(
        audio,
        num_mel_bins=80,  # 梅尔频带数
        dither=0,         # 禁用抖动
        sample_frequency=16000
    )

    # 特征归一化
    feat = feat - feat.mean(dim=0, keepdim=True)

    # 配置ONNX Runtime会话
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)

    # 使用ONNX模型进行推理
    input_name = ort_session.get_inputs()[0].name
    embedding = ort_session.run(None, {input_name: feat.unsqueeze(dim=0).cpu().numpy()})[0]

    # 返回 embedding
    return embedding.flatten()

In [4]:
# 提取 speaker embedding
for idx, audio_path in enumerate(audio_paths):
    embedding = extract_embedding(audio_path)
    embedding = torch.tensor(embedding).unsqueeze(0) # [1, 192]

Machat 提取 Speaker fealture Mel

In [7]:
import torch
import sys

sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
from matcha.utils.audio import mel_spectrogram

# sampling_rate = 22050

class Feat_extractor:

    def __init__(self, n_fft: int = 1920, num_mels: int = 80, sampling_rate: int = 24000, 
                 hop_size: int = 480, win_size: int = 1920, fmin: int = 0, fmax: int = 8000, 
                 center: bool = False, device: str = 'cuda'):
        self.device = torch.device(device)
        self.feat_extractor = lambda x: mel_spectrogram(
            x, n_fft=n_fft, num_mels=num_mels, sampling_rate=sampling_rate, 
            hop_size=hop_size, win_size=win_size, fmin=fmin, fmax=fmax, center=center
        )

    def __call__(self, speech):
        speech_feat = self.feat_extractor(speech).squeeze(dim=0).transpose(0, 1).to(self.device)
        speech_feat = speech_feat.unsqueeze(dim=0)
        speech_feat_len = torch.tensor([speech_feat.shape[1]], dtype=torch.int32).to(self.device)
        return speech_feat, speech_feat_len


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import sys
sys.path.append('/root/LLMs/CosyVoice')

from cosyvoice.utils.file_utils import load_wav

# 提取 mel 特征
front_end = Feat_extractor()

resample_rate = 22050
# 提取 speaker embedding
for idx, audio_path in enumerate(audio_paths):
    input_speech_16k = load_wav(audio_path, resample_rate)
    input_feat, input_feat_len = front_end(input_speech_16k) # [1, 452, 80]

GLM 批量提取 speech token

In [1]:
import sys
import os
sys.path.append('/root/LLMs/GLM-4-Voice')

import torch
from flow_inference import AudioDecoder
import os.path


flow_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
tokenizer_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"

flow_config = os.path.join(flow_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(flow_path, "hift.pt")
device = "cuda"
audio_decoder: AudioDecoder = None
whisper_model, feature_extractor = None, None

sys.path.insert(0, "/root/LLMs/GLM-4-Voice/cosyvoice")
sys.path.insert(0, "/root/LLMs/CosyVoice/third_party/Matcha-TTS")

# Flow & Hift
audio_decoder = AudioDecoder(config_path=flow_config,
                             flow_ckpt_path=flow_checkpoint,
                             hift_ckpt_path=hift_checkpoint,
                             device=device,)


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/_jit_internal.py:739: FutureWarning: ignore(True) has been deprecated. TorchScript will now drop the function call on compilation. Use torch.jit.unused now. {}
  warnings.warn(
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weigh

In [ ]:
import sys
import os
sys.path.append('/root/LLMs/GLM-4-Voice')

tokenizer_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"
device = "cuda"

from speech_tokenizer.modeling_whisper import WhisperVQEncoder
from transformers import WhisperFeatureExtractor
from speech_tokenizer.utils import extract_speech_token


# Speech tokenizer
whisper_model = (WhisperVQEncoder.from_pretrained(tokenizer_path).eval().to(device))

feature_extractor = WhisperFeatureExtractor.from_pretrained(tokenizer_path)

# 提取音频中的 token。utts 是一个音频路径 list
audio_tokens = extract_speech_token(whisper_model, feature_extractor, utts)

# audio_tokens = torch.tensor(audio_tokens) # [B, T]

/root/miniconda3/envs/lingC/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# pipeline

In [5]:
import os
import sys

# 添加路径到系统
sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
sys.path.append('/root/LLMs/GLM-4-Voice')
sys.path.append('/root/LLMs/GLM-4-Voice/cosyvoice')


import json
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import onnxruntime

from matcha.utils.audio import mel_spectrogram
from cosyvoice.utils.file_utils import load_wav
from flow_inference import AudioDecoder
from speech_tokenizer.modeling_whisper import WhisperVQEncoder
from transformers import WhisperFeatureExtractor
from speech_tokenizer.utils import extract_speech_token


# 配置路径
json_path = '/root/LLMs/data/instruction_300_text.json'
audio_dir = '/root/LLMs/data/glm_flow'
onnx_model_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/campplus.onnx'
flow_path = "/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
tokenizer_path = "/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"
device = "cuda"

# 初始化模型
flow_config = os.path.join(flow_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(flow_path, "hift.pt")
audio_decoder = AudioDecoder(config_path=flow_config, flow_ckpt_path=flow_checkpoint, hift_ckpt_path=hift_checkpoint, device=device)
whisper_model = WhisperVQEncoder.from_pretrained(tokenizer_path).eval().to(device)
feature_extractor = WhisperFeatureExtractor.from_pretrained(tokenizer_path)

# 生成音频文件路径
def generate_audio_paths(json_path, audio_dir):
    """
    根据 JSON 文件中的顺序生成对应的音频文件路径列表
    :param json_path: JSON 文件路径
    :param audio_dir: 音频文件目录
    :return: 音频文件路径列表
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    audio_paths = [os.path.join(audio_dir, f"Q_{idx}_0.wav") for idx in range(len(data))]
    return audio_paths

# 提取 embedding
def extract_embedding(audio_path, onnx_model_path):
    """
    提取单个音频文件的 embedding
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: embedding (numpy array)
    """
    audio, sample_rate = torchaudio.load(audio_path)
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)
    feat = kaldi.fbank(audio, num_mel_bins=80, dither=0, sample_frequency=16000)
    feat = feat - feat.mean(dim=0, keepdim=True)
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)
    input_name = ort_session.get_inputs()[0].name
    embedding = ort_session.run(None, {input_name: feat.unsqueeze(dim=0).cpu().numpy()})[0]
    return embedding.flatten()

# 提取 mel 特征
class Feat_extractor:
    def __init__(self, n_fft=1920, num_mels=80, sampling_rate=24000, hop_size=480, win_size=1920, fmin=0, fmax=8000, center=False, device='cuda'):
        self.device = torch.device(device)
        self.feat_extractor = lambda x: mel_spectrogram(
            x, n_fft=n_fft, num_mels=num_mels, sampling_rate=sampling_rate, hop_size=hop_size, win_size=win_size, fmin=fmin, fmax=fmax, center=center
        )

    def __call__(self, speech):
        speech_feat = self.feat_extractor(speech)
        speech_feat = speech_feat
        speech_feat_len = torch.tensor([speech_feat.shape[-1]], dtype=torch.int32)
        return speech_feat, speech_feat_len

# 提取 speech token
def extract_GLM_speech_token(audio_paths):
    """
    提取音频中的 speech token
    :param audio_paths: 音频文件路径列表
    :return: speech token
    """
    return extract_speech_token(whisper_model, feature_extractor, audio_paths)

# 主流程
def main(json_path, audio_dir, onnx_model_path, device='cuda'):
    """
    主流程：从 JSON 和音频文件中提取所有数据
    :param json_path: JSON 文件路径
    :param audio_dir: 音频文件目录
    :param onnx_model_path: ONNX 模型路径
    :param device: 设备（cuda 或 cpu）
    :return: 包含所有数据的列表
    """
    # 生成音频文件路径
    audio_paths = generate_audio_paths(json_path, audio_dir)
    audio_paths = audio_paths # 仅处理前 3 个文件

    # 从 JSON 文件中提取 text
    with open(json_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
    texts = [item['output'] for item in json_data]  # 提取所有 "output" 字段

    # 初始化结果列表
    data = []

    # 初始化特征提取器
    front_end = Feat_extractor(sampling_rate=22050, device=device)

    # 遍历每个音频文件
    total_files = len(audio_paths)
    for idx, audio_path in enumerate(audio_paths):
        if not os.path.exists(audio_path):
            print(f"Warning: Audio file {audio_path} not found. Skipping.")
            continue

        # 显示进度
        print(f"Processing file {idx + 1}/{total_files}: {audio_path}")

        # 提取 embedding
        embedding = extract_embedding(audio_path, onnx_model_path)
        embedding = torch.tensor(embedding)
        # 提取 mel 特征
        input_speech = load_wav(audio_path, 22050)
        speech_feat, speech_feat_len = front_end(input_speech)

        # 提取 speech token
        audio_tokens = extract_GLM_speech_token([audio_path])
        audio_tokens = torch.tensor(audio_tokens)

        # 构造样本
        sample = {
            'id': idx,
            'text': texts[idx],  # 从 JSON 中提取的 text
            'speech_token': audio_tokens.squeeze(0),
            'speech_token_len': torch.tensor([audio_tokens.shape[1]], dtype=torch.long),
            'speech_feat': speech_feat,
            'speech_feat_len': speech_feat_len,
            'embedding': embedding.unsqueeze(0)
        }

        # 添加到结果列表
        data.append(sample)

    return data

# 运行主流程
data = main(json_path, audio_dir, onnx_model_path, device)

# 保存数据为 .pt 文件
output_pt_path = '/root/LLMs/data/processed_data.pt'
torch.save(data, output_pt_path)
print(f"Data saved to {output_pt_path}")

Processing file 1/302: /root/LLMs/data/glm_flow/Q_0_0.wav
Processing file 2/302: /root/LLMs/data/glm_flow/Q_1_0.wav
Processing file 3/302: /root/LLMs/data/glm_flow/Q_2_0.wav
Processing file 4/302: /root/LLMs/data/glm_flow/Q_3_0.wav
Processing file 5/302: /root/LLMs/data/glm_flow/Q_4_0.wav
Processing file 6/302: /root/LLMs/data/glm_flow/Q_5_0.wav
Processing file 7/302: /root/LLMs/data/glm_flow/Q_6_0.wav
Processing file 8/302: /root/LLMs/data/glm_flow/Q_7_0.wav
Processing file 9/302: /root/LLMs/data/glm_flow/Q_8_0.wav
Processing file 10/302: /root/LLMs/data/glm_flow/Q_9_0.wav
Processing file 11/302: /root/LLMs/data/glm_flow/Q_10_0.wav
Processing file 12/302: /root/LLMs/data/glm_flow/Q_11_0.wav
Processing file 13/302: /root/LLMs/data/glm_flow/Q_12_0.wav
Processing file 14/302: /root/LLMs/data/glm_flow/Q_13_0.wav
Processing file 15/302: /root/LLMs/data/glm_flow/Q_14_0.wav
Processing file 16/302: /root/LLMs/data/glm_flow/Q_15_0.wav
Processing file 17/302: /root/LLMs/data/glm_flow/Q_16_0.wav

In [14]:
data[2]

{'id': 2,
 'text': '我是纳友科技设计的助手，可以帮助您处理各种任务。',
 'speech_token': tensor([[15416, 11149,  7661,  2573, 10045,  3472, 15312,  3044,  4559,  5518,
           7143, 15043,  6590, 15063, 11139,   164,  2098,  6240,  4363,  7315,
           9249,  1758, 10693,  8065,  5312,  2585, 14039,  6691,  9914,  8321,
          14247, 12436,  1791, 11124,  2585,  8745,  8381, 11034,  6963, 10045,
           5492,  5492, 15526,  6240,  6940,  4794,  8064, 13889, 14129,  9409,
          10483, 14260,   485,  4992, 14440,   659,  3700, 15526,  1035,  8374,
           1176, 15669,  7584,  4804,   260,  6614,  3954,  5312, 13251,  5703,
           3608, 15885, 13524,  4280,  3558,  6259, 10644,   623,  2066]]),
 'speech_token_len': tensor([79]),
 'speech_feat': tensor([[-4.7151, -4.1129, -2.7735,  ..., -8.9640, -9.4289, -9.4896],
         [-4.3725, -4.2290, -2.7482,  ..., -9.2786, -9.3869, -9.5783],
         [-4.4471, -4.3472, -3.6836,  ..., -9.1224, -9.4186, -9.7294],
         ...,
         [-4.8273, -5.24

In [17]:
data[2]['embedding'].shape

torch.Size([192])

测试

## safetensor转pt


In [1]:
import torch
from safetensors.torch import load_file, save_file

# 加载 .safetensors 文件
safetensors_path = "/root/LLMs/models/glm_flow/checkpoint-380/model.safetensors"
state_dict = load_file(safetensors_path)

# 保存为 .pt 文件
pt_path = "/root/LLMs/models/glm_flow/checkpoint-380/model.pt"
torch.save(state_dict, pt_path)

print(f"模型已成功转换为 {pt_path}")

模型已成功转换为 /root/LLMs/models/glm_flow/checkpoint-380/model.pt


In [24]:
import sys
import os
sys.path.append('/root/LLMs/GLM-4-Voice')

import torch
from flow_inference import AudioDecoder
import os.path

# flow_path="/root/LLMs/models/glm_flow/checkpoint-380"
flow_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
hifigan_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
tokenizer_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"

flow_config = os.path.join(hifigan_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(hifigan_path, "hift.pt")
device = "cuda"
audio_decoder: AudioDecoder = None
whisper_model, feature_extractor = None, None

sys.path.insert(0, "/root/LLMs/GLM-4-Voice/cosyvoice")
sys.path.insert(0, "/root/LLMs/CosyVoice/third_party/Matcha-TTS")

# Flow & Hift
audio_decoder = AudioDecoder(config_path=flow_config,
                             flow_ckpt_path=flow_checkpoint,
                             hift_ckpt_path=hift_checkpoint,
                             device=device,)

import uuid
# 生成唯一标识符
this_uuid = str(uuid.uuid4())

In [3]:
data = torch.load('/root/LLMs/data/processed_data.pt')

In [4]:
data[2]

{'id': 2,
 'text': '我是纳友科技设计的助手，可以帮助您处理各种任务。',
 'speech_token': tensor([15416, 11149,  7661,  2573, 10045,  3472, 15312,  3044,  4559,  5518,
          7143, 15043,  6590, 15063, 11139,   164,  2098,  6240,  4363,  7315,
          9249,  1758, 10693,  8065,  5312,  2585, 14039,  6691,  9914,  8321,
         14247, 12436,  1791, 11124,  2585,  8745,  8381, 11034,  6963, 10045,
          5492,  5492, 15526,  6240,  6940,  4794,  8064, 13889, 14129,  9409,
         10483, 14260,   485,  4992, 14440,   659,  3700, 15526,  1035,  8374,
          1176, 15669,  7584,  4804,   260,  6614,  3954,  5312, 13251,  5703,
          3608, 15885, 13524,  4280,  3558,  6259, 10644,   623,  2066]),
 'speech_token_len': tensor([79]),
 'speech_feat': tensor([[[-4.5813, -4.3026, -4.3657,  ..., -4.8050, -4.6137, -4.3823],
          [-3.6911, -3.7411, -4.1008,  ..., -5.1676, -4.9583, -4.7631],
          [-2.6540, -2.6553, -3.7187,  ..., -5.0046, -5.3912, -5.4060],
          ...,
          [-9.4731, -9.5459, 

In [5]:
audio_tokens = data[2]['speech_token']

input_feat = data[2]['speech_feat']

embedding = data[2]['embedding']

In [21]:
audio_tokens = torch.tensor(x['Q_51_0'])

In [25]:
# 生成语音和梅尔特征
with torch.no_grad():
    # 转换为张量
    tts_speech, tts_mel = audio_decoder.token2wav(
        token=audio_tokens.unsqueeze(0),
        # prompt_token = audio_tokens,
        uuid=this_uuid,
        # prompt_feat=input_feat.transpose(1, 2),
        # embedding=embedding,
        finalize=True
    )

import IPython.display as ipd

# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 22050
tts_speech = tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)

加载 flow 模型

In [ ]:
import sys
sys.path.append('/root/LLMs/GLM-4-Voice')
sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')

from hyperpyyaml import load_hyperpyyaml
config_path = '/root/LLMs/models/glm_flow/config.yaml'

with open(config_path, 'r') as f:
    scratch_configs = load_hyperpyyaml(f)


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


In [10]:
import torch
flow = scratch_configs['flow']
device = 'cuda'

flow.load_state_dict(torch.load('/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder/flow.pt', map_location=device))

<All keys matched successfully>

In [1]:
import torchaudio

audio, sample_rate = torchaudio.load('/root/LLMs/data/glm_flow/Q_0_0.wav')

In [2]:
sample_rate

22050

In [3]:

import IPython.display as ipd

# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 22050
tts_speech = audio.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)

In [26]:
import torch 

x = torch.load('/root/LLMs/data/processed_data.pt')